# 01 -- CFTC Commitment of Traders Data Collection

## Source
CFTC Public Reporting Socrata API (`https://publicreporting.cftc.gov/resource/gpe5-46if.json`), querying the **Traders in Financial Futures (TFF)** report for **S&P 500 E-Mini Futures only** (CFTC contract market code `13874A`).

## Collection Method
Records are fetched in batches of 5,000 via paginated GET requests, ordered by `report_date_as_yyyy_mm_dd ASC`, with a 0.5-second sleep between requests. All batches are concatenated into a single DataFrame.

## Variables Collected
Raw position columns pulled from the API (note: the API uses inconsistent suffixes -- Dealer columns end with `_all`, while Leveraged, Asset Manager, and Other Reportable columns do not):

- **Leveraged Funds:** `lev_money_positions_long`, `lev_money_positions_short`, `lev_money_positions_spread`
- **Asset Manager / Institutional:** `asset_mgr_positions_long`, `asset_mgr_positions_short`, `asset_mgr_positions_spread`
- **Dealer / Intermediary:** `dealer_positions_long_all`, `dealer_positions_short_all`, `dealer_positions_spread_all`
- **Other Reportables:** `other_rept_positions_long`, `other_rept_positions_short`, `other_rept_positions_spread`
- **Open Interest:** `open_interest_all`

## Derived Factors
- **Net positions:** `lev_net`, `am_net`, `dealer_net` (long minus short for each trader category)
- **Net positions as % of open interest:** `lev_net_pct`, `am_net_pct`, `dealer_net_pct`
- **Leveraged-to-Asset-Manager ratio:** `lev_am_ratio` (lev_net / am_net, with zero replaced by NaN)
- **Week-over-week changes:** `lev_net_chg`, `am_net_chg` (first differences of net positions)

## Timing Adjustment
An `available_date` column is created as `date + 6 days` (the Monday after the release Friday) to reflect the date at which the data would actually be available to a trader, preventing look-ahead bias.

## Filtering
Data is filtered to dates on or before 2024-12-31.

## Output
`Data/Data_Collection/Initial/01_CTFC/01_cftc.parquet`

In [ ]:
"""
CFTC Commitment of Traders — Traders in Financial Futures (TFF)
S&P 500 E-Mini Futures Only (CFTC Code 13874A)
"""

import pandas as pd
import numpy as np
from pathlib import Path
import requests
import time



print("=" * 80)
print("CFTC TFF — S&P 500 E-MINI FUTURES (Code 13874A)")
print("=" * 80)

BASE_URL = "https://publicreporting.cftc.gov/resource/gpe5-46if.json"

all_records = []
offset = 0
BATCH = 5000

print("Fetching from CFTC Socrata API...")
while True:
    params = {
        "$where": "cftc_contract_market_code='13874A'",
        "$limit": BATCH,
        "$offset": offset,
        "$order": "report_date_as_yyyy_mm_dd ASC",
    }
    resp = requests.get(BASE_URL, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    if not data:
        break
    all_records.extend(data)
    print(f"  Fetched {len(all_records)} records...")
    offset += BATCH
    time.sleep(0.5)

print(f"Total raw records: {len(all_records)}")

cot_raw = pd.DataFrame(all_records)

# ── Map raw API columns → clean names ────────────────────────────────────────
# IMPORTANT: The CFTC Socrata API has INCONSISTENT naming:
#   - Dealer columns end with '_all'
#   - Leveraged, Asset Manager, Other Reportable do NOT have '_all'
# These names are taken directly from the API output above.

pos_cols = {
    # Leveraged Funds — NO '_all' suffix
    "lev_money_positions_long":    "lev_long",
    "lev_money_positions_short":   "lev_short",
    "lev_money_positions_spread":  "lev_spread",
    # Asset Manager / Institutional — NO '_all' suffix
    "asset_mgr_positions_long":    "am_long",
    "asset_mgr_positions_short":   "am_short",
    "asset_mgr_positions_spread":  "am_spread",
    # Dealer / Intermediary — HAS '_all' suffix
    "dealer_positions_long_all":   "dealer_long",
    "dealer_positions_short_all":  "dealer_short",
    "dealer_positions_spread_all": "dealer_spread",
    # Other Reportables — NO '_all' suffix
    "other_rept_positions_long":   "other_long",
    "other_rept_positions_short":  "other_short",
    "other_rept_positions_spread": "other_spread",
    # Open interest — HAS '_all' suffix
    "open_interest_all":           "open_interest",
}

cot = pd.DataFrame()
cot["date"] = pd.to_datetime(cot_raw["report_date_as_yyyy_mm_dd"])

missing_cols = []
for src, dst in pos_cols.items():
    if src in cot_raw.columns:
        cot[dst] = pd.to_numeric(cot_raw[src], errors="coerce")
    else:
        missing_cols.append(src)
        cot[dst] = np.nan

if missing_cols:
    print(f"\n⚠ STILL MISSING: {missing_cols}")
else:
    print("\n✓ All position columns found successfully")

# ── Derived factors ──────────────────────────────────────────────────────────
cot["lev_net"]    = cot["lev_long"]    - cot["lev_short"]
cot["am_net"]     = cot["am_long"]     - cot["am_short"]
cot["dealer_net"] = cot["dealer_long"] - cot["dealer_short"]

cot["lev_net_pct"]    = cot["lev_net"]    / cot["open_interest"] * 100
cot["am_net_pct"]     = cot["am_net"]     / cot["open_interest"] * 100
cot["dealer_net_pct"] = cot["dealer_net"] / cot["open_interest"] * 100

cot["lev_am_ratio"] = cot["lev_net"] / cot["am_net"].replace(0, np.nan)

cot = cot.sort_values("date").reset_index(drop=True)
cot["lev_net_chg"] = cot["lev_net"].diff()
cot["am_net_chg"]  = cot["am_net"].diff()

# ── available_date = Monday after release Friday (date + 6 days) ─────────────
cot["available_date"] = cot["date"] + pd.Timedelta(days=6)

# ── Filter ───────────────────────────────────────────────────────────────────
cot = cot[cot["date"] <= "2024-12-31"].reset_index(drop=True)

# ── Save ─────────────────────────────────────────────────────────────────────
cot.to_parquet("../../Data/Data_Collection/Initial/01_CTFC/01_cftc.parquet", index=False, engine="pyarrow")

print(f"\nSaved data/cftc_sp500_cot.parquet: {cot.shape}")
print(f"Date range: {cot['date'].min().date()} → {cot['date'].max().date()}")
print(f"Rows: {len(cot)}")

factor_cols = [c for c in cot.columns if c not in ["date", "available_date"]]
print(f"\nFactors ({len(factor_cols)}):")
for i, c in enumerate(factor_cols, 1):
    n_valid = cot[c].notna().sum()
    print(f"  {i:>2d}. {c:<20s} — {n_valid:>4d} valid obs")

CFTC TFF — S&P 500 E-MINI FUTURES (Code 13874A)
Fetching from CFTC Socrata API...
  Fetched 1038 records...
Total raw records: 1038

✓ All position columns found successfully

Saved data/cftc_sp500_cot.parquet: (969, 24)
Date range: 2006-06-13 → 2024-12-31
Rows: 969

Factors (22):
   1. lev_long             —  969 valid obs
   2. lev_short            —  969 valid obs
   3. lev_spread           —  969 valid obs
   4. am_long              —  969 valid obs
   5. am_short             —  969 valid obs
   6. am_spread            —  969 valid obs
   7. dealer_long          —  969 valid obs
   8. dealer_short         —  969 valid obs
   9. dealer_spread        —  969 valid obs
  10. other_long           —  969 valid obs
  11. other_short          —  969 valid obs
  12. other_spread         —  969 valid obs
  13. open_interest        —  969 valid obs
  14. lev_net              —  969 valid obs
  15. am_net               —  969 valid obs
  16. dealer_net           —  969 valid obs
  17. lev_net_